<a href="https://colab.research.google.com/github/Swag-Pseudopy/Conic-Particle-Methods-for-MNE/blob/main/CPMDA_Slingshot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# cpmda_sling_fixed.py
# KEY FIXES:
# 1. Position step uses PRE-UPDATE weights (saved before weight update)
# 2. Adaptive clipping: clip is proportional to weight, so heavy particles move less
# 3. More iterations + larger particle count for better mass collapse visibility
# 4. Added a rich convergence dashboard (4-panel) with entropy, Hamiltonian, and barycenter dist

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# ==========================================
# 1. Slingshot Stepsize Scheduler
# ==========================================
def get_slingshot_steps(problem_type, t, T, L=None, m=None, M=None, rng=None):
    rng = rng or np.random.default_rng()

    if problem_type == "bilinear":
        k = t // 2
        r_t = (M + m) / 2 + (M - m) / 2 * np.cos((2 * k + 1) * np.pi / (2 * T))
        h = 1.0 / np.sqrt(r_t)
        return (h, -h) if t % 2 == 0 else (-h, h)
    else:
        h = 1.0 / (3.0 * L)
        if t % 2 == 0:
            if rng.random() < 0.5:
                return h, -h
            else:
                return -h, h
        else:
            return h, h

# ==========================================
# 2. Conic Particle MDA Algorithm
# ==========================================
class ConicParticles:
    def __init__(self, n, m, d, rng=None):
        rng = rng or np.random.default_rng()
        self.a = np.ones(n) / n
        self.x = rng.uniform(-2, 2, (n, d))
        self.b = np.ones(m) / m
        self.y = rng.uniform(-2, 2, (m, d))

    def update_mda(self, grad_a, grad_x, grad_b, grad_y, alpha, beta):
        # FIX 1: Save weights BEFORE updating them
        a_old = self.a.copy()
        b_old = self.b.copy()

        # ---- Player 1 (Descent) ----
        # Weight Update (Log-Sum-Exp Trick)
        log_a = np.log(np.maximum(a_old, 1e-15)) - alpha * grad_a
        log_a -= np.max(log_a)
        self.a = np.exp(log_a)
        self.a /= np.sum(self.a)

        # FIX 2: Position update uses PRE-update weights (a_old), not self.a
        # FIX 3: Adaptive clipping — max step scales with weight so ghost particles
        #         can still explore but heavy particles are conservative
        denom_x = np.maximum(a_old[:, None], 1e-3)
        step_x = alpha * grad_x / denom_x
        # Clip per-particle: heavy particle (a~1/n initially) gets tighter clip
        clip_val = np.minimum(1.0, 0.1 / np.maximum(a_old[:, None], 1e-3))
        self.x -= np.clip(step_x, -clip_val, clip_val)

        # ---- Player 2 (Ascent) ----
        log_b = np.log(np.maximum(b_old, 1e-15)) + beta * grad_b
        log_b -= np.max(log_b)
        self.b = np.exp(log_b)
        self.b /= np.sum(self.b)

        denom_y = np.maximum(b_old[:, None], 1e-3)
        step_y = beta * grad_y / denom_y
        clip_val_y = np.minimum(1.0, 0.1 / np.maximum(b_old[:, None], 1e-3))
        self.y += np.clip(step_y, -clip_val_y, clip_val_y)

        self.x = np.clip(self.x, -2.5, 2.5)
        self.y = np.clip(self.y, -2.5, 2.5)

# ==========================================
# 3. Objective Functions & Gradients
# ==========================================
def f_bilinear(x, y, B):   return x @ B @ y
def grad_bilinear(x, y, B): return B @ y, B.T @ x

def logcosh(x):
    s = np.sign(x) * x
    p = np.exp(-2 * s)
    return s + np.log1p(p) - np.log(2.0)

def f_convex_concave(x, y):    return float(logcosh(x) - logcosh(y))
def grad_convex_concave(x, y): return np.tanh(x), -np.tanh(y)

def f_sc_sc(x, y, mu=1.0):    return float(0.5 * mu * np.sum(x**2) - 0.5 * mu * np.sum(y**2) + np.sum(x * y))
def grad_sc_sc(x, y, mu=1.0): return mu * x + y, -mu * y + x

# ==========================================
# 4. Diagnostics helpers
# ==========================================
def get_weighted_barycenter(weights, positions):
    return np.sum(weights[:, None] * positions, axis=0)

def entropy(weights):
    w = np.maximum(weights, 1e-15)
    return -np.sum(w * np.log(w))

def duality_gap(a, x, b, y, f_func):
    """
    Approximate exploitability / duality gap.
    max_j sum_i a_i f(x_i, y_j)  -  min_i sum_j b_j f(x_i, y_j)
    """
    vals = np.array([[float(f_func(x[i], y[j])) for j in range(len(y))] for i in range(len(x))])
    # P1's value against best P2 response
    v_p2_best = np.max(vals.T @ a)   # max_j  sum_i a_i f(xi, yj)
    # P2's value against best P1 response
    v_p1_best = np.min(vals @ b)     # min_i  sum_j b_j f(xi, yj)
    return v_p2_best - v_p1_best     # >= 0; zero iff MNE

# ==========================================
# 5. Experiment runner
# ==========================================
def run_experiment(model, grad_func, f_func, steps=200,
                   problem_type="bilinear", T=None, L=None, m=None, M=None,
                   rng=None):
    rng = rng or np.random.default_rng()
    T = T or steps
    history = {'a': [], 'x': [], 'b': [], 'y': [],
               'barycenter': [], 'entropy_a': [], 'entropy_b': [],
               'hamiltonian': [], 'gap': []}

    for t in range(steps):
        history['a'].append(model.a.copy())
        history['x'].append(model.x.copy())
        history['b'].append(model.b.copy())
        history['y'].append(model.y.copy())

        xb = get_weighted_barycenter(model.a, model.x)[0]
        yb = get_weighted_barycenter(model.b, model.y)[0]
        history['barycenter'].append((xb, yb))
        history['entropy_a'].append(entropy(model.a))
        history['entropy_b'].append(entropy(model.b))

        gx0, gy0 = grad_func(np.array([xb]), np.array([yb]))
        history['hamiltonian'].append(0.5 * (float(np.sum(gx0**2)) + float(np.sum(gy0**2))))

        # Duality gap every 10 steps (expensive O(n*m) call)
        if t % 10 == 0:
            history['gap'].append(duality_gap(model.a, model.x, model.b, model.y, f_func))
        else:
            history['gap'].append(np.nan)

        alpha, beta = get_slingshot_steps(problem_type, t, T, L=L, m=m, M=M, rng=rng)

        n_, m_ = model.x.shape[0], model.y.shape[0]
        grad_x = np.zeros_like(model.x)
        grad_y = np.zeros_like(model.y)
        grad_a = np.zeros(n_)
        grad_b = np.zeros(m_)

        for i in range(n_):
            for j in range(m_):
                gx, gy = grad_func(model.x[i], model.y[j])
                val = float(f_func(model.x[i], model.y[j]))
                grad_x[i] += model.b[j] * float(gx)
                grad_y[j] += model.a[i] * float(gy)
                grad_a[i] += model.b[j] * val
                grad_b[j] += model.a[i] * val

        model.update_mda(grad_a, grad_x, grad_b, grad_y, alpha, beta)

    return history

# ==========================================
# 6. Rich 4-panel static convergence dashboard
# ==========================================
def plot_dashboard(history, grad_func, title=""):
    """
    Panel 1: Hamiltonian landscape + barycenter path
    Panel 2: Log barycenter distance to origin
    Panel 3: Entropy of both players (should DROP to 0 at convergence)
    Panel 4: Duality gap (should DROP to 0 at convergence)
    """
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # --- Panel 1: Landscape ---
    xs = np.linspace(-3, 3, 80)
    X, Y = np.meshgrid(xs, xs)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            gx, gy = grad_func(np.array([X[i, j]]), np.array([Y[i, j]]))
            Z[i, j] = 0.5 * (float(np.sum(gx**2)) + float(np.sum(gy**2)))
    axs[0].contourf(X, Y, Z, levels=30, cmap='Blues', alpha=0.5)

    traj = np.array(history['barycenter'])
    mask = ~np.isnan(traj).any(axis=1)
    tc = traj[mask]
    if len(tc) > 0:
        axs[0].plot(tc[:, 0], tc[:, 1], 'k-', lw=2, label='Barycenter', zorder=3)
        axs[0].scatter(*tc[0], c='white', edgecolors='k', s=100, zorder=4, label='Start')
        axs[0].scatter(*tc[-1], c='yellow', edgecolors='k', marker='*', s=400, zorder=5, label='End')
    a_f, x_f = history['a'][-1], history['x'][-1]
    b_f, y_f = history['b'][-1], history['y'][-1]
    axs[0].scatter(x_f[:, 0], np.full(len(x_f), -0.05),
                   s=np.maximum(a_f * 2000, 5), c='red', alpha=0.8, edgecolors='k', zorder=5)
    axs[0].scatter(np.full(len(y_f), 0.05), y_f[:, 0],
                   s=np.maximum(b_f * 2000, 5), c='orange', alpha=0.8, edgecolors='k', zorder=5)
    axs[0].set_xlim(-3, 3); axs[0].set_ylim(-3, 3)
    axs[0].set_title("Hamiltonian + Particles"); axs[0].legend(fontsize=7)

    # --- Panel 2: Log barycenter distance ---
    if len(tc) > 0:
        dist = np.sqrt(tc[:, 0]**2 + tc[:, 1]**2)
        axs[1].semilogy(dist, c='purple', lw=2)
        axs[1].set_title("Barycenter Distance (log)")
        axs[1].set_xlabel("Iteration"); axs[1].set_ylabel("||bary||")
        axs[1].grid(True, ls='--', alpha=0.5)

    # --- Panel 3: Entropy ---
    # At MNE, entropy should → 0 (all mass on one particle = pure strategy)
    ea = np.array(history['entropy_a'])
    eb = np.array(history['entropy_b'])
    axs[2].plot(ea, c='red', lw=2, label='H(a) - P1')
    axs[2].plot(eb, c='orange', lw=2, label='H(b) - P2')
    axs[2].axhline(0, color='black', ls='--', lw=1, label='Target (0)')
    axs[2].set_title("Strategy Entropy (↓ = mass collapse)")
    axs[2].set_xlabel("Iteration"); axs[2].set_ylabel("Entropy")
    axs[2].legend(fontsize=8); axs[2].grid(True, ls='--', alpha=0.5)

    # --- Panel 4: Duality gap ---
    gap = np.array(history['gap'])
    valid_gap = ~np.isnan(gap)
    iters = np.where(valid_gap)[0]
    axs[3].semilogy(iters, gap[valid_gap], 'g-o', lw=2, ms=4, label='Duality Gap')
    axs[3].axhline(0.01, color='red', ls='--', lw=1, label='ε=0.01')
    axs[3].set_title("Duality Gap (↓ = Nash)")
    axs[3].set_xlabel("Iteration"); axs[3].set_ylabel("Gap (log)")
    axs[3].legend(fontsize=8); axs[3].grid(True, ls='--', alpha=0.5)

    plt.tight_layout()
    return fig

# ==========================================
# 7. GIF Generator (same as before, now includes entropy panel)
# ==========================================
def animate_convergence(history, grad_func, filename="convergence.gif", title=""):
    fig, axs = plt.subplots(1, 4, figsize=(20, 5))

    xs = np.linspace(-3, 3, 80)
    X, Y = np.meshgrid(xs, xs)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            gx, gy = grad_func(np.array([X[i, j]]), np.array([Y[i, j]]))
            Z[i, j] = 0.5 * (float(np.sum(gx**2)) + float(np.sum(gy**2)))
    axs[0].contourf(X, Y, Z, levels=30, cmap='Blues', alpha=0.4)

    scat_p1 = axs[0].scatter([], [], c='red', edgecolors='black', zorder=5)
    scat_p2 = axs[0].scatter([], [], c='orange', edgecolors='black', zorder=4)
    line_bary, = axs[0].plot([], [], 'k-', lw=2, zorder=2)
    scat_bary = axs[0].scatter([], [], c='yellow', edgecolors='black', marker='*', s=300, zorder=10)
    axs[0].set_xlim(-3, 3); axs[0].set_ylim(-3, 3)
    axs[0].set_title(f"{title}")

    steps_range = np.arange(len(history['a']))
    ea = np.array(history['entropy_a'])
    eb = np.array(history['entropy_b'])

    def update(frame):
        a, x = history['a'][frame], history['x'][frame]
        b, y = history['b'][frame], history['y'][frame]

        scat_p1.set_offsets(np.c_[x[:, 0], np.full(len(x), -0.05)])
        scat_p1.set_sizes(np.maximum(a * 1500, 5))
        scat_p2.set_offsets(np.c_[np.full(len(y), 0.05), y[:, 0]])
        scat_p2.set_sizes(np.maximum(b * 1500, 5))

        bary_path = np.array(history['barycenter'][:frame+1])
        line_bary.set_data(bary_path[:, 0], bary_path[:, 1])
        scat_bary.set_offsets([bary_path[-1]])

        # P1 mass histogram
        axs[1].clear()
        axs[1].set_xlim(-3, 3); axs[1].set_ylim(0, 1.1)
        axs[1].set_title(f"P1 Mass (H={ea[frame]:.2f})")
        axs[1].bar(x[:, 0], a, width=0.15, color='red', alpha=0.7, edgecolor='k')

        # P2 mass histogram
        axs[2].clear()
        axs[2].set_xlim(-3, 3); axs[2].set_ylim(0, 1.1)
        axs[2].set_title(f"P2 Mass (H={eb[frame]:.2f})")
        axs[2].bar(y[:, 0], b, width=0.15, color='orange', alpha=0.7, edgecolor='k')

        # Entropy over time
        axs[3].clear()
        axs[3].plot(steps_range[:frame+1], ea[:frame+1], 'r-', lw=1.5, label='H(a)')
        axs[3].plot(steps_range[:frame+1], eb[:frame+1], color='orange', lw=1.5, label='H(b)')
        axs[3].set_xlim(0, len(history['a']))
        axs[3].set_ylim(-0.1, np.log(len(a)) + 0.2)
        axs[3].set_title("Strategy Entropy")
        axs[3].set_xlabel("t"); axs[3].legend(fontsize=7)
        axs[3].grid(True, ls='--', alpha=0.4)

    print(f"Generating {filename}...")
    ani = animation.FuncAnimation(fig, update, frames=len(history['a']), interval=80)
    ani.save(filename, writer='pillow')
    plt.close(fig)
    print("Done!")

# ==========================================
# 8. Driver
# ==========================================
if __name__ == "__main__":
    rng = np.random.default_rng(42)

    # ---- Case 1: Bilinear ----
    print("=== Bilinear (CP-MDA fixed) ===")
    B_mat = np.array([[1.0]])
    model_bl = ConicParticles(n=30, m=30, d=1, rng=rng)
    history_bl = run_experiment(
        model_bl,
        lambda x, y: grad_bilinear(x, y, B_mat),
        lambda x, y: f_bilinear(x, y, B_mat),
        steps=120, problem_type="bilinear", m=1.0, M=1.0, rng=rng
    )
    fig = plot_dashboard(history_bl, lambda x, y: grad_bilinear(x, y, B_mat),
                         title="Bilinear - CP-MDA Fixed")
    fig.savefig("cpmda_bilinear_dashboard.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    animate_convergence(history_bl, lambda x, y: grad_bilinear(x, y, B_mat),
                        filename="cpmda_bilinear.gif", title="Bilinear CP-MDA")

    # ---- Case 2: Convex-Concave ----
    print("\n=== Convex-Concave (CP-MDA fixed) ===")
    model_cc = ConicParticles(n=30, m=30, d=1, rng=rng)
    history_cc = run_experiment(
        model_cc, grad_convex_concave, f_convex_concave,
        steps=200, problem_type="nonlinear", L=1.0, rng=rng
    )
    fig = plot_dashboard(history_cc, grad_convex_concave,
                         title="Convex-Concave - CP-MDA Fixed")
    fig.savefig("cpmda_cc_dashboard.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    animate_convergence(history_cc, grad_convex_concave,
                        filename="cpmda_cc.gif", title="Conv-Conc CP-MDA")

    # ---- Case 3: SC-SC ----
    print("\n=== SC-SC (CP-MDA fixed) ===")
    model_sc = ConicParticles(n=30, m=30, d=1, rng=rng)
    history_sc = run_experiment(
        model_sc,
        lambda x, y: grad_sc_sc(x, y, mu=1.0),
        lambda x, y: f_sc_sc(x, y, mu=1.0),
        steps=150, problem_type="nonlinear", L=np.sqrt(2), rng=rng
    )
    fig = plot_dashboard(history_sc, lambda x, y: grad_sc_sc(x, y, mu=1.0),
                         title="SC-SC - CP-MDA Fixed")
    fig.savefig("cpmda_scsc_dashboard.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    animate_convergence(history_sc, lambda x, y: grad_sc_sc(x, y, mu=1.0),
                        filename="cpmda_scsc.gif", title="SC-SC CP-MDA")

    print("\nAll done!")

=== Bilinear (CP-MDA fixed) ===


/tmp/ipykernel_12484/1830588731.py:163: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  grad_x[i] += model.b[j] * float(gx)
/tmp/ipykernel_12484/1830588731.py:164: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  grad_y[j] += model.a[i] * float(gy)


Generating cpmda_bilinear.gif...
Done!

=== Convex-Concave (CP-MDA fixed) ===


/tmp/ipykernel_12484/1830588731.py:90: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  def f_convex_concave(x, y):    return float(logcosh(x) - logcosh(y))
/tmp/ipykernel_12484/1830588731.py:163: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  grad_x[i] += model.b[j] * float(gx)
/tmp/ipykernel_12484/1830588731.py:164: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  grad_y[j] += model.a[i] * float(gy)


Generating cpmda_cc.gif...
Done!

=== SC-SC (CP-MDA fixed) ===


/tmp/ipykernel_12484/1830588731.py:163: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  grad_x[i] += model.b[j] * float(gx)
/tmp/ipykernel_12484/1830588731.py:164: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  grad_y[j] += model.a[i] * float(gy)


Generating cpmda_scsc.gif...
Done!

All done!


!pip install pillow

import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Slingshot Stepsize Scheduler
# ==========================================
def get_slingshot_steps(problem_type, t, T, L=None, m=None, M=None, rng=None):
    rng = rng or np.random.default_rng()

    if problem_type == "bilinear":
        # Chebyshev-based schedule for Linear Gradients
        k = t // 2
        r_t = (M + m) / 2 + (M - m) / 2 * np.cos((2 * k + 1) * np.pi / (2 * T))
        h = 1.0 / np.sqrt(r_t)
        return (h, -h) if t % 2 == 0 else (-h, h)

    else:
        # Randomized symmetry breaking for Nonlinear Gradients
        h = 1.0 / (3.0 * L)
        if t % 2 == 0:
            if rng.random() < 0.5:
                return h, -h
            else:
                return -h, h
        else:
            return h, h

# ==========================================
# 2. Conic Particle MDA Algorithm (STABLE)
# ==========================================
class ConicParticles:
    def __init__(self, n, m, d, rng=None):
        rng = rng or np.random.default_rng()
        self.a = np.ones(n) / n
        self.x = rng.uniform(-2, 2, (n, d))
        self.b = np.ones(m) / m
        self.y = rng.uniform(-2, 2, (m, d))

    def update_mda(self, grad_a, grad_x, grad_b, grad_y, alpha, beta):
        # ---- Player 1 (Descent) ----
        # 1. Stable Weight Update (Log-Sum-Exp Trick)
        log_a = np.log(np.maximum(self.a, 1e-15)) - alpha * grad_a
        log_a -= np.max(log_a) # Prevent overflow before exp()
        self.a = np.exp(log_a)
        self.a /= np.sum(self.a)

        # 2. Stable Position Update (Strict Physical Bounding)
        # Cap denominator at 1e-3, and strictly limit the max physical jump to 0.5 units
        step_x = alpha * grad_x / np.maximum(self.a[:, None], 1e-3)
        self.x -= np.clip(step_x, -0.5, 0.5)

        # ---- Player 2 (Ascent) ----
        # 1. Stable Weight Update (Log-Sum-Exp Trick)
        log_b = np.log(np.maximum(self.b, 1e-15)) + beta * grad_b
        log_b -= np.max(log_b)
        self.b = np.exp(log_b)
        self.b /= np.sum(self.b)

        # 2. Stable Position Update (Strict Physical Bounding)
        step_y = beta * grad_y / np.maximum(self.b[:, None], 1e-3)
        self.y += np.clip(step_y, -0.5, 0.5)

# ==========================================
# 3. Objective Functions & Gradients
# ==========================================
def f_bilinear(x, y, B):
    return x @ B @ y

def grad_bilinear(x, y, B):
    return B @ y, B.T @ x

# STABILITY FIX: Stable log(cosh)
def logcosh(x):
    s = np.sign(x) * x
    p = np.exp(-2 * s)
    return s + np.log1p(p) - np.log(2.0)

def f_convex_concave(x, y):
    return logcosh(x) - logcosh(y)

def grad_convex_concave(x, y):
    return np.tanh(x), -np.tanh(y)

def f_sc_sc(x, y, mu=1.0):
    return 0.5 * mu * np.sum(x**2) - 0.5 * mu * np.sum(y**2) + np.sum(x * y)

def grad_sc_sc(x, y, mu=1.0):
    return mu * x + y, -mu * y + x

# ==========================================
# 4. Plotting & Diagnostics
# ==========================================
def get_weighted_barycenter(weights, positions):
    """Computes the true expected strategy by weighting positions by probabilities."""
    return np.sum(weights[:, None] * positions, axis=0)

def plot_advanced_diagnostics(grad_func, traj, model, title=""):
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    # ---- Plot 1: Hamiltonian Landscape & Particles ----
    xs, ys = np.linspace(-3, 3, 100), np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)

    # Compute Hamiltonian (0.5 * ||grad f||^2) for the background
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            gx, gy = grad_func(np.array([X[i, j]]), np.array([Y[i, j]]))
            Z[i, j] = 0.5 * (np.sum(gx**2) + np.sum(gy**2))

    axs[0].contourf(X, Y, Z, levels=30, cmap='Blues', alpha=0.5, zorder=1)

    # NaN Filter Safety Net
    traj = np.array(traj)
    valid_mask = ~np.isnan(traj).any(axis=1)
    traj_clean = traj[valid_mask]

    if len(traj_clean) > 0:
        # Plot Barycenter Trajectory
        axs[0].plot(traj_clean[:, 0], traj_clean[:, 1], c='black', linestyle='-', linewidth=2, label='Barycenter Path', zorder=2)
        axs[0].scatter(traj_clean[0, 0], traj_clean[0, 1], c='white', edgecolors='black', marker='o', s=100, label='Start', zorder=3)

        # Visually distinct, large End marker on top of everything
        axs[0].scatter(traj_clean[-1, 0], traj_clean[-1, 1], c='yellow', edgecolors='black', marker='*', s=400, label='End (Eq)', zorder=10)

    # VISIBILITY FIX: Plot Final Particles
    # Add tiny offsets so particles don't hide on the axis gridlines
    p1_y_offset = np.full_like(model.x[:, 0], -0.1)
    p2_x_offset = np.full_like(model.y[:, 0], 0.1)

    # Scale by weight, but enforce a minimum size so they don't vanish entirely
    s_p1 = np.maximum(model.a * 1500, 10)
    s_p2 = np.maximum(model.b * 1500, 10)

    axs[0].scatter(model.x[:, 0], p1_y_offset,
                   s=s_p1, c='red', alpha=0.8, edgecolors='black', label='P1 Particles (x)', zorder=5)
    axs[0].scatter(p2_x_offset, model.y[:, 0],
                   s=s_p2, c='orange', alpha=0.8, edgecolors='black', label='P2 Particles (y)', zorder=4)

    axs[0].set_xlim(-3, 3)
    axs[0].set_ylim(-3, 3)
    axs[0].set_title(f"{title} - Hamiltonian Landscape")
    axs[0].legend(loc='upper right', fontsize=8)
    axs[0].grid(True, linestyle='--', alpha=0.5, zorder=0)

    # ---- Plot 2: Convergence Metrics ----
    if len(traj_clean) > 0:
        dist_to_origin = np.sqrt(traj_clean[:, 0]**2 + traj_clean[:, 1]**2)
        axs[1].plot(dist_to_origin, label='Barycenter Dist to (0,0)', c='purple', linewidth=2)
        axs[1].set_yscale('log')
        axs[1].set_title("Convergence over Time")
        axs[1].set_xlabel("Iteration")
        axs[1].set_ylabel("Log Distance")
        axs[1].legend()
        axs[1].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

# # ==========================================
# # 5. Experiment Runner
# ==========================================
# def run_experiment(model, grad_func, f_func, steps=100, problem_type="bilinear", T=None, L=None, m=None, M=None):
#     trajectory = []
#     T = T or steps

#     for t in range(steps):
#         # 1. Log the starting/current Barycenter
#         x_bar = get_weighted_barycenter(model.a, model.x)[0]
#         y_bar = get_weighted_barycenter(model.b, model.y)[0]
#         trajectory.append((x_bar, y_bar))

#         # 2. Get Steps
#         alpha, beta = get_slingshot_steps(problem_type, t, T, L=L, m=m, M=M)

#         # 3. Compute Gradients
#         n, m_ = model.x.shape[0], model.y.shape[0]
#         grad_x, grad_y = np.zeros_like(model.x), np.zeros_like(model.y)
#         grad_a, grad_b = np.zeros(n), np.zeros(m_)

#         for i in range(n):
#             for j in range(m_):
#                 gx, gy = grad_func(model.x[i], model.y[j])
#                 val = f_func(model.x[i], model.y[j])

#                 # Use float() to extract scalar and prevent NumPy warnings
#                 grad_x[i] += model.b[j] * float(gx)
#                 grad_y[j] += model.a[i] * float(gy)
#                 grad_a[i] += model.b[j] * float(val)
#                 grad_b[j] += model.a[i] * float(val)

#         # 4. Update MDA
#         model.update_mda(grad_a, grad_x, grad_b, grad_y, alpha, beta)

#     return trajectory

# # ==========================================
# # 6. Execute Scenarios
# # ==========================================
# if __name__ == "__main__":
#     rng = np.random.default_rng(42) # Fixed seed for stable visualization

#     print("Running Bilinear...")
#     B = np.array([[1.0]])
#     model_bl = ConicParticles(n=20, m=20, d=1, rng=rng)
#     traj_bl = run_experiment(model_bl, lambda x, y: grad_bilinear(x, y, B), lambda x, y: f_bilinear(x, y, B),
#                              steps=60, problem_type="bilinear", m=1.0, M=1.0)
#     plot_advanced_diagnostics(lambda x, y: grad_bilinear(x, y, B), traj_bl, model_bl, title="Bilinear (Slingshot CP-MDA)")

#     print("Running Convex-Concave...")
#     model_cc = ConicParticles(n=20, m=20, d=1, rng=rng)
#     traj_cc = run_experiment(model_cc, grad_convex_concave, f_convex_concave,
#                              steps=150, problem_type="nonlinear", L=1.0)
#     plot_advanced_diagnostics(grad_convex_concave, traj_cc, model_cc, title="Convex-Concave (Slingshot CP-MDA)")

#     print("Running SC-SC...")
#     model_sc = ConicParticles(n=20, m=20, d=1, rng=rng)
#     traj_sc = run_experiment(model_sc, lambda x, y: grad_sc_sc(x, y, mu=1.0), lambda x, y: f_sc_sc(x, y, mu=1.0),
#                              steps=100, problem_type="nonlinear", L=np.sqrt(2))
#     plot_advanced_diagnostics(lambda x, y: grad_sc_sc(x, y, mu=1.0), traj_sc, model_sc, title="SC-SC (Slingshot CP-MDA)")

import matplotlib.animation as animation

# ==========================================
# 1. Updated Experiment Runner (Captures History)
# ==========================================
def run_experiment_with_history(model, grad_func, f_func, steps=100, problem_type="bilinear", T=None, L=None, m=None, M=None):
    T = T or steps

    # Dictionary to store the full state at every step
    history = {
        'a': [], 'x': [],
        'b': [], 'y': [],
        'barycenter': []
    }

    for t in range(steps):
        # Record Current State (Use .copy() to capture the state, not the reference)
        history['a'].append(model.a.copy())
        history['x'].append(model.x.copy())
        history['b'].append(model.b.copy())
        history['y'].append(model.y.copy())

        x_bar = get_weighted_barycenter(model.a, model.x)[0]
        y_bar = get_weighted_barycenter(model.b, model.y)[0]
        history['barycenter'].append((x_bar, y_bar))

        # Get Steps and Compute Gradients (Standard CP-MDA logic)
        alpha, beta = get_slingshot_steps(problem_type, t, T, L=L, m=m, M=M)
        n, m_ = model.x.shape[0], model.y.shape[0]
        grad_x, grad_y = np.zeros_like(model.x), np.zeros_like(model.y)
        grad_a, grad_b = np.zeros(n), np.zeros(m_)

        for i in range(n):
            for j in range(m_):
                gx, gy = grad_func(model.x[i], model.y[j])
                val = f_func(model.x[i], model.y[j])
                grad_x[i] += model.b[j] * float(gx)
                grad_y[j] += model.a[i] * float(gy)
                grad_a[i] += model.b[j] * float(val)
                grad_b[j] += model.a[i] * float(val)

        # Update
        model.update_mda(grad_a, grad_x, grad_b, grad_y, alpha, beta)

    return history

# ==========================================
# 2. The GIF Generator
# ==========================================
def animate_convergence(history, grad_func, filename="convergence.gif", title=""):
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # --- Setup Left Plot: 2D Landscape ---
    xs, ys = np.linspace(-3, 3, 100), np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            gx, gy = grad_func(np.array([X[i, j]]), np.array([Y[i, j]]))
            Z[i, j] = 0.5 * (np.sum(gx**2) + np.sum(gy**2))
    axs[0].contourf(X, Y, Z, levels=30, cmap='Blues', alpha=0.4)

    # Initialize scatter plots for the animation
    scat_p1 = axs[0].scatter([], [], c='red', edgecolors='black', zorder=5, label='P1 Particles')
    scat_p2 = axs[0].scatter([], [], c='orange', edgecolors='black', zorder=4, label='P2 Particles')
    line_bary, = axs[0].plot([], [], c='black', linestyle='-', linewidth=2, zorder=2)
    scat_bary = axs[0].scatter([], [], c='yellow', edgecolors='black', marker='*', s=300, zorder=10)

    axs[0].set_xlim(-3, 3)
    axs[0].set_ylim(-3, 3)
    axs[0].set_title(f"{title} - State Space")

    # --- Setup Middle & Right Plots: Probability Distributions ---
    axs[1].set_xlim(-3, 3)
    axs[1].set_ylim(0, 1.1)
    axs[1].set_title("P1 Strategy Mass ($a_i$ vs $x_i$)")
    axs[1].set_xlabel("Position (x)")
    axs[1].set_ylabel("Probability Mass")

    axs[2].set_xlim(-3, 3)
    axs[2].set_ylim(0, 1.1)
    axs[2].set_title("P2 Strategy Mass ($b_j$ vs $y_j$)")
    axs[2].set_xlabel("Position (y)")

    # The update function called for each frame
    def update(frame):
        # 1. Update Landscape Plot
        a, x = history['a'][frame], history['x'][frame]
        b, y = history['b'][frame], history['y'][frame]

        # Format offsets and sizes
        scat_p1.set_offsets(np.c_[x[:, 0], np.full_like(x[:, 0], -0.1)])
        scat_p1.set_sizes(np.maximum(a * 1000, 10))

        scat_p2.set_offsets(np.c_[np.full_like(y[:, 0], 0.1), y[:, 0]])
        scat_p2.set_sizes(np.maximum(b * 1000, 10))

        bary_path = np.array(history['barycenter'][:frame+1])
        if len(bary_path) > 0:
            line_bary.set_data(bary_path[:, 0], bary_path[:, 1])
            scat_bary.set_offsets(bary_path[-1])

        # 2. Update Probability Bar Charts
        axs[1].clear()
        axs[1].set_xlim(-3, 3)
        axs[1].set_ylim(0, 1.1)
        axs[1].set_title("P1 Strategy Mass ($a_i$ vs $x_i$)")
        axs[1].bar(x[:, 0], a, width=0.15, color='red', alpha=0.7, edgecolor='black')

        axs[2].clear()
        axs[2].set_xlim(-3, 3)
        axs[2].set_ylim(0, 1.1)
        axs[2].set_title("P2 Strategy Mass ($b_j$ vs $y_j$)")
        axs[2].bar(y[:, 0], b, width=0.15, color='orange', alpha=0.7, edgecolor='black')

    print(f"Generating {filename}...")
    ani = animation.FuncAnimation(fig, update, frames=len(history['a']), interval=100)
    ani.save(filename, writer='pillow')
    plt.close(fig)
    print("Done!")

# ==========================================
# 3. Driver
# ==========================================
if __name__ == "__main__":
    rng = np.random.default_rng(42) # Seed for consistency

    # ==========================================
    # Case 1: Bilinear (B = [[1.0]])
    # ==========================================
    print("Running Bilinear Animation...")
    B_mat = np.array([[1.0]])
    model_bl = ConicParticles(n=20, m=20, d=1, rng=rng)

    history_bl = run_experiment_with_history(
        model_bl,
        lambda x, y: grad_bilinear(x, y, B_mat),
        lambda x, y: f_bilinear(x, y, B_mat),
        steps=60, problem_type="bilinear", m=1.0, M=1.0
    )

    animate_convergence(
        history_bl,
        lambda x, y: grad_bilinear(x, y, B_mat),
        filename="cpmda_bilinear.gif",
        title="Bilinear (Slingshot CP-MDA)"
    )

    # ==========================================
    # Case 2: Convex-Concave (log-cosh)
    # ==========================================
    print("\nRunning Convex-Concave Animation...")
    model_cc = ConicParticles(n=20, m=20, d=1, rng=rng)

    history_cc = run_experiment_with_history(
        model_cc,
        grad_convex_concave,
        f_convex_concave,
        steps=150, problem_type="nonlinear", L=1.0
    )

    animate_convergence(
        history_cc,
        grad_convex_concave,
        filename="cpmda_convex_concave.gif",
        title="Convex-Concave (Slingshot CP-MDA)"
    )

    # ==========================================
    # Case 3: Strongly Convex-Strongly Concave
    # ==========================================
    print("\nRunning SC-SC Animation...")
    model_sc = ConicParticles(n=20, m=20, d=1, rng=rng)

    history_sc = run_experiment_with_history(
        model_sc,
        lambda x, y: grad_sc_sc(x, y, mu=1.0),
        lambda x, y: f_sc_sc(x, y, mu=1.0),
        steps=100, problem_type="nonlinear", L=np.sqrt(2)
    )

    animate_convergence(
        history_sc,
        lambda x, y: grad_sc_sc(x, y, mu=1.0),
        filename="cpmda_scsc.gif",
        title="SC-SC (Slingshot CP-MDA)"
    )

    print("\nAll animations generated successfully!")